# 🚀 ConicTypst - Sổ Tay AI Chuyển Đổi Đề Thi Hàng Loạt (Gemini Vision + Pandoc)

Chuyển đổi toàn bộ các file PDF đề thi sang **mã nguồn Typst chuẩn `@preview/sang-math:1.0.5`** và **file Word `.docx` (Equation OMML)** hoàn toàn miễn phí trên Kaggle.

---
### ⚡ Hướng dẫn: Chạy lần lượt từ BƯỚC 1 đến BƯỚC 4.

In [ ]:
# ==============================================================================
# CONICTYPST ALL-IN-ONE: TỰ ĐỘNG LẤY KEY VĨNH VIỄN & QUÉT ĐỀ THI TOÁN SANG TYPST
# ==============================================================================
import os, sys, glob, time, subprocess

# 🔑 TỰ ĐỘNG NẠP KEY TỪ KAGGLE SECRETS
try:
    from kaggle_secrets import UserSecretsClient
    GEMINI_API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = "AQ.Ab8RN6IvY9r6dnT9Q82bjymH9-ug7B-VlBnCYfeb8u-vr8VO-g"

print("🔑 Đã nạp API Key vĩnh viễn sẵn sàng!\n", flush=True)

# 1. CÀI ĐẶT THƯ VIỆN & PANDOC
print("⏳ BƯỚC 1/3: Chuẩn bị môi trường & Pandoc...", flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymupdf", "google-genai", "pillow", "ipywidgets"], check=True)
subprocess.run("apt-get update -qq && apt-get install -y -qq pandoc", shell=True, check=True)
print("✅ Môi trường đã sẵn sàng 100%!\n", flush=True)

import fitz, zipfile
from google import genai
from google.genai import types
from IPython.display import display, HTML, FileLink
import ipywidgets as widgets

client = genai.Client(api_key=GEMINI_API_KEY.strip())
MODELS_CHAIN = ['gemini-3.5-flash', 'gemini-3.7-flash', 'gemini-3.8-flash']

PROMPT_SANG_MATH = """Bạn là chuyên gia chuyển đổi đề thi Toán THPT Việt Nam thành mã nguồn Typst chính xác 100%.
QUY TẮC BẮT BUỘC:
1. Sử dụng gói chính thức: #import "@preview/sang-math:1.0.5": *
2. Bốn định dạng câu hỏi chuẩn Bộ Giáo Dục:
   - Trắc nghiệm 4 lựa chọn (Phần I):
     #tn([Nội dung câu hỏi], ([$A$], True([$B$]), [$C$], [$D$]), id: "TN01", loigiai: [Lời giải chi tiết])
   - Trắc nghiệm Đúng - Sai (Phần II):
     #ds([Nội dung chung], (True([Mệnh đề a đúng]), [Mệnh đề b sai], True([Mệnh đề c đúng]), [Mệnh đề d sai]), id: "DS01", loigiai: [Lời giải chi tiết])
   - Trả lời ngắn (Phần III):
     #tln([Nội dung câu hỏi], [$42$], id: "TLN01", loigiai: [Lời giải chi tiết])
   - Tự luận (Phần IV):
     #tl([Nội dung câu hỏi], id: "TL01", lines: 6, loigiai: [Lời giải chi tiết])
3. QUY TẮC VẼ HÌNH HỌC:
   - Nếu trong đề có hình vẽ: Ưu tiên dựng bằng CeTZ kết hợp các hàm sm-* chuẩn của sang-math-geom.
4. ĐỊNH DẠNG ĐẦU RA:
   - Chỉ trả về MÃ NGUỒN TYPST DUY NHẤT, không bọc trong ```typst hay văn bản thừa."""

def ocr_page_gemini_smart(image_bytes):
    for m in MODELS_CHAIN:
        for attempt in range(1, 4):
            try:
                res = client.models.generate_content(
                    model=m,
                    contents=[
                        types.Part.from_bytes(data=image_bytes, mime_type="image/jpeg"),
                        "Hãy nhận diện toàn bộ câu hỏi và công thức toán trong trang đề thi này thành mã Typst chuẩn sang-math:1.0.5."
                    ],
                    config=types.GenerateContentConfig(
                        system_instruction=PROMPT_SANG_MATH,
                        temperature=0.1,
                        max_output_tokens=8192,
                        http_options=types.HttpOptions(timeout=50000)
                    )
                )
                txt = res.text or ""
                return txt.replace("```typst", "").replace("```", "").strip()
            except Exception as e:
                err = str(e)
                if "429" in err or "RESOURCE_EXHAUSTED" in err:
                    print(f" [Đợi 15s hồi lượt 429...]", end="", flush=True)
                    time.sleep(15)
                elif "503" in err or "UNAVAILABLE" in err:
                    print(f" [{m} bận, đổi model...]", end="", flush=True)
                    time.sleep(2)
                    break
                else:
                    time.sleep(2)
    return "// [Lỗi: Không thể nhận diện trang này]"

def run_batch_process():
    os.makedirs('output_typst', exist_ok=True)
    os.makedirs('output_docx', exist_ok=True)
    all_pdfs = sorted(glob.glob('input_pdf/*.pdf')) or sorted(glob.glob('*.pdf'))
    all_pdfs = [p for p in all_pdfs if not p.startswith('.') and os.path.isfile(p)]
    
    if not all_pdfs:
        print("⚠️ Chưa tìm thấy file PDF nào trong danh sách tải lên!", flush=True)
        return
        
    print(f"\n🎯 BƯỚC 2/3: Bắt đầu xử lý {len(all_pdfs)} file đề thi: {[os.path.basename(p) for p in all_pdfs]}", flush=True)
    
    for pdf_path in all_pdfs:
        doc = fitz.open(pdf_path)
        fname = os.path.basename(pdf_path).replace('.pdf', '')
        print(f"\n🚀 Đang quét file: {fname}.pdf ({len(doc)} trang)...", flush=True)
        
        header = f"""// ================================================================
// ĐỀ THI TOÁN THPT - CHUẨN SANG-MATH:1.0.5
// Nhận diện tự động qua ConicTypst Kaggle Batch Engine
// ================================================================

#import "@preview/sang-math:1.0.5": *

#show: exam-theme.with(
  paper: "a4",
  cols: 2,
  theme: "ocean",
  show-header: true,
  show-footer: true,
  watermark: none,
  watermark-opacity: 0.05,
  school: [TRƯỜNG THPT],
  title: [{fname}],
  subject: [Môn: Toán - Thời gian: 90 phút],
  exam-id: "001"
)

"""
        typ_file = os.path.join('output_typst', f'{fname}.typ')
        with open(typ_file, 'w', encoding='utf-8') as f:
            f.write(header)
            
        for i, page in enumerate(doc):
            print(f"  -> Nhận diện trang {i+1}/{len(doc)}...", end="", flush=True)
            pix = page.get_pixmap(dpi=180)
            code = ocr_page_gemini_smart(pix.tobytes("jpeg"))
            code = code.replace('#import "@preview/sang-math:1.0.5": *', '')
            page_content = f"\n// --- NỘI DUNG TỪ TRANG {i+1} ---\n" + code + "\n"
            
            with open(typ_file, 'a', encoding='utf-8') as f:
                f.write(page_content)
            print(" [XONG]", flush=True)
            time.sleep(4.5)
            
        print(f"  ✅ Đã xuất Typst an toàn: {typ_file}", flush=True)
        
        docx_file = os.path.join('output_docx', f'{fname}.docx')
        try:
            subprocess.run(['pandoc', '-f', 'markdown', '-t', 'docx', typ_file, '-o', docx_file], check=True, timeout=60)
            print(f"  ✅ Đã xuất Word DOCX: {docx_file}", flush=True)
        except Exception as err:
            pass

    print("\n📦 BƯỚC 3/3: Đang đóng gói file Zip...", flush=True)
    zip_filename = 'dethi_conictypst_hoan_thanh.zip'
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for typ in glob.glob('output_typst/*.typ'):
            zipf.write(typ, os.path.join('Typst_Sources', os.path.basename(typ)))
        for docx in glob.glob('output_docx/*.docx'):
            zipf.write(docx, os.path.join('Word_Docx', os.path.basename(docx)))

    print('=======================================================', flush=True)
    print('🎉 TẤT CẢ ĐỀ THI ĐÃ HOÀN TẤT! BẤM LINK DƯỚI ĐỂ TẢI VỀ MÁY:', flush=True)
    display(FileLink(zip_filename))

# Màn hình hiển thị kết quả (Bắt buộc trong ipywidgets)
output_screen = widgets.Output()
uploader = widgets.FileUpload(accept='.pdf', multiple=True, description='📁 Chọn file PDF', button_style='primary')
btn_start = widgets.Button(description='🚀 BẮT ĐẦU QUÉT ĐỀ', button_style='success')

def on_start_click(b):
    btn_start.disabled = True
    btn_start.description = "⏳ ĐANG XỬ LÝ..."
    btn_start.button_style = "warning"
    with output_screen:
        output_screen.clear_output()
        print("⚡ ĐÃ NHẬN LỆNH! Đang nạp file PDF...", flush=True)
        os.makedirs('input_pdf', exist_ok=True)
        val = uploader.value
        items = val if isinstance(val, (list, tuple)) else val.values() if isinstance(val, dict) else []
        count = 0
        for item in items:
            name = item.get('name', 'exam.pdf') if isinstance(item, dict) else getattr(item, 'name', 'exam.pdf')
            content = item.get('content', b'') if isinstance(item, dict) else getattr(item, 'content', b'')
            with open(os.path.join('input_pdf', name), 'wb') as f:
                f.write(content)
            count += 1
            print(f"  📥 Đã nạp file: {name}", flush=True)
        
        run_batch_process()
        
    btn_start.disabled = False
    btn_start.description = "🚀 BẮT ĐẦU QUÉT ĐỀ"
    btn_start.button_style = "success"

btn_start.on_click(on_start_click)

# Hiển thị nút bấm và màn hình kết quả
print("👉 Thầy bấm [📁 Chọn file PDF] rồi bấm nút xanh [🚀 BẮT ĐẦU QUÉT ĐỀ] nhé:")
display(widgets.HBox([uploader, btn_start]))
display(output_screen)



In [ ]:
# ========================================================
# BƯỚC 3: Nhận diện bằng Gemini 3.5 / 3.7 / 3.8 Flash & Xuất Typst + Word
# ========================================================
import glob, os, time, subprocess, fitz
from google import genai
from google.genai import types

os.makedirs('output_typst', exist_ok=True)
os.makedirs('output_docx', exist_ok=True)

all_pdfs = sorted(glob.glob('input_pdf/*.pdf')) or sorted(glob.glob('*.pdf'))
if not all_pdfs:
    raise ValueError('⚠️ Chưa có file PDF nào. Thầy vui lòng bấm nút Chọn file ở BƯỚC 2 trước nhé!')

print(f'🎯 Đã nhận diện {len(all_pdfs)} file PDF: {[os.path.basename(p) for p in all_pdfs]}', flush=True)

client = genai.Client(api_key=GEMINI_API_KEY.strip())

PROMPT_SANG_MATH = '''Bạn là chuyên gia chuyển đổi đề thi Toán THPT Việt Nam thành mã nguồn Typst chính xác 100%.
QUY TẮC BẮT BUỘC:
1. Sử dụng gói chính thức: #import "@preview/sang-math:1.0.5": *
2. Bốn định dạng câu hỏi chuẩn Bộ Giáo Dục:
   - Trắc nghiệm 4 lựa chọn (Phần I):
     #tn([Nội dung câu hỏi], ([$A$], True([$B$]), [$C$], [$D$]), id: "TN01", loigiai: [Lời giải chi tiết])
   - Trắc nghiệm Đúng - Sai (Phần II):
     #ds([Nội dung chung], (True([Mệnh đề a đúng]), [Mệnh đề b sai], True([Mệnh đề c đúng]), [Mệnh đề d sai]), id: "DS01", loigiai: [Lời giải chi tiết])
   - Trả lời ngắn (Phần III):
     #tln([Nội dung câu hỏi], [$42$], id: "TLN01", loigiai: [Lời giải chi tiết])
   - Tự luận (Phần IV):
     #tl([Nội dung câu hỏi], id: "TL01", lines: 6, loigiai: [Lời giải chi tiết])
3. QUY TẮC VẼ HÌNH HỌC:
   - Nếu trong đề có hình vẽ: Ưu tiên dựng bằng CeTZ kết hợp các hàm sm-* chuẩn của sang-math-geom.
4. ĐỊNH DẠNG ĐẦU RA:
   - Chỉ trả về MÃ NGUỒN TYPST DUY NHẤT, không bọc trong ```typst hay văn bản thừa.'''

# CHỈ SỬ DỤNG DUY NHẤT CÁC MODEL 3.5, 3.7 VÀ 3.8 FLASH
MODELS_CHAIN = ['gemini-3.5-flash', 'gemini-3.7-flash', 'gemini-3.8-flash']

def ocr_page_gemini(image_bytes):
    for m in MODELS_CHAIN:
        for attempt in range(1, 4):
            try:
                res = client.models.generate_content(
                    model=m,
                    contents=[
                        types.Part.from_bytes(data=image_bytes, mime_type='image/jpeg'),
                        'Hãy nhận diện toàn bộ câu hỏi và công thức toán trong trang đề thi này thành mã Typst chuẩn sang-math:1.0.5.'
                    ],
                    config=types.GenerateContentConfig(
                        system_instruction=PROMPT_SANG_MATH,
                        temperature=0.1,
                        max_output_tokens=8192,
                        http_options=types.HttpOptions(timeout=50000)
                    )
                )
                txt = res.text or ''
                return txt.replace('```typst', '').replace('```', '').strip()
            except Exception as e:
                err_msg = str(e)
                if '429' in err_msg or 'RESOURCE_EXHAUSTED' in err_msg:
                    print(f' [{m} chạm quota 429, nghỉ 15s hồi lượt...]', end='', flush=True)
                    time.sleep(15)
                elif '503' in err_msg or 'UNAVAILABLE' in err_msg:
                    print(f' [{m} 503 bận, thử model tiếp theo...]', end='', flush=True)
                    time.sleep(3)
                    break
                else:
                    print(f' [{m} lỗi, thử lại...]', end='', flush=True)
                    time.sleep(3)
    return '// [Lỗi: Không thể nhận diện trang này]'

for pdf_path in all_pdfs:
    doc = fitz.open(pdf_path)
    fname = os.path.basename(pdf_path).replace('.pdf', '')
    print(f'\n🚀 Bắt đầu nhận diện: {fname}.pdf ({len(doc)} trang)...', flush=True)
    
    header = f'''// ================================================================
// ĐỀ THI TOÁN THPT - CHUẨN SANG-MATH:1.0.5
// Nhận diện tự động qua ConicTypst Kaggle Batch Engine
// ================================================================

#import "@preview/sang-math:1.0.5": *

#show: exam-theme.with(
  paper: "a4",
  cols: 2,
  theme: "ocean",
  show-header: true,
  show-footer: true,
  watermark: none,
  watermark-opacity: 0.05,
  school: [TRƯỜNG THPT],
  title: [{fname}],
  subject: [Môn: Toán - Thời gian: 90 phút],
  exam-id: "001"
)

'''
    typ_file = os.path.join('output_typst', f'{fname}.typ')
    with open(typ_file, 'w', encoding='utf-8') as f:
        f.write(header)
        
    for i, page in enumerate(doc):
        print(f'  -> Nhận diện trang {i+1}/{len(doc)}...', end='', flush=True)
        pix = page.get_pixmap(dpi=180)
        code = ocr_page_gemini(pix.tobytes('jpeg'))
        code = code.replace('#import "@preview/sang-math:1.0.5": *', '')
        page_content = f'\n// --- NỘI DUNG TỪ TRANG {i+1} ---\n' + code + '\n'
        
        with open(typ_file, 'a', encoding='utf-8') as f:
            f.write(page_content)
        print(' [XONG]', flush=True)
        
        # Nghỉ 4 giây giữa các trang để KHÔNG BỊ 429 trên các model 3.x Flash
        time.sleep(4)
        
    print(f'  ✅ Đã xuất Typst an toàn: {typ_file}', flush=True)
    
    docx_file = os.path.join('output_docx', f'{fname}.docx')
    try:
        subprocess.run([
            'pandoc', '-f', 'markdown', '-t', 'docx',
            typ_file, '-o', docx_file
        ], check=True, timeout=60)
        print(f'  ✅ Đã xuất Word DOCX: {docx_file}', flush=True)
    except Exception as err:
        pass

print('\n🎉 BƯỚC 3 ĐÃ HOÀN TẤT TẤT CẢ FILE ĐỀ THI!', flush=True)

In [ ]:
# ========================================================
# BƯỚC 3: Nhận diện bằng Gemini 3.5 / 3.7 / 3.8 Flash & Xuất Typst + Word
# ========================================================
import glob, os, time, subprocess, fitz
from google import genai
from google.genai import types

os.makedirs('output_typst', exist_ok=True)
os.makedirs('output_docx', exist_ok=True)

all_pdfs = sorted(glob.glob('input_pdf/*.pdf')) or sorted(glob.glob('*.pdf'))
if not all_pdfs:
    raise ValueError('⚠️ Chưa có file PDF nào. Thầy vui lòng bấm nút Chọn file ở BƯỚC 2 trước nhé!')

print(f'🎯 Đã nhận diện {len(all_pdfs)} file PDF: {[os.path.basename(p) for p in all_pdfs]}', flush=True)

client = genai.Client(api_key=GEMINI_API_KEY.strip())

PROMPT_SANG_MATH = '''Bạn là chuyên gia chuyển đổi đề thi Toán THPT Việt Nam thành mã nguồn Typst chính xác 100%.
QUY TẮC BẮT BUỘC:
1. Sử dụng gói chính thức: #import "@preview/sang-math:1.0.5": *
2. Bốn định dạng câu hỏi chuẩn Bộ Giáo Dục:
   - Trắc nghiệm 4 lựa chọn (Phần I):
     #tn([Nội dung câu hỏi], ([$A$], True([$B$]), [$C$], [$D$]), id: "TN01", loigiai: [Lời giải chi tiết])
   - Trắc nghiệm Đúng - Sai (Phần II):
     #ds([Nội dung chung], (True([Mệnh đề a đúng]), [Mệnh đề b sai], True([Mệnh đề c đúng]), [Mệnh đề d sai]), id: "DS01", loigiai: [Lời giải chi tiết])
   - Trả lời ngắn (Phần III):
     #tln([Nội dung câu hỏi], [$42$], id: "TLN01", loigiai: [Lời giải chi tiết])
   - Tự luận (Phần IV):
     #tl([Nội dung câu hỏi], id: "TL01", lines: 6, loigiai: [Lời giải chi tiết])
3. QUY TẮC VẼ HÌNH HỌC:
   - Nếu trong đề có hình vẽ: Ưu tiên dựng bằng CeTZ kết hợp các hàm sm-* chuẩn của sang-math-geom.
4. ĐỊNH DẠNG ĐẦU RA:
   - Chỉ trả về MÃ NGUỒN TYPST DUY NHẤT, không bọc trong ```typst hay văn bản thừa.'''

# CHỈ SỬ DỤNG DUY NHẤT CÁC MODEL 3.5, 3.7 VÀ 3.8 FLASH
MODELS_CHAIN = ['gemini-3.5-flash', 'gemini-3.7-flash', 'gemini-3.8-flash']

def ocr_page_gemini(image_bytes):
    for m in MODELS_CHAIN:
        for attempt in range(1, 4):
            try:
                res = client.models.generate_content(
                    model=m,
                    contents=[
                        types.Part.from_bytes(data=image_bytes, mime_type='image/jpeg'),
                        'Hãy nhận diện toàn bộ câu hỏi và công thức toán trong trang đề thi này thành mã Typst chuẩn sang-math:1.0.5.'
                    ],
                    config=types.GenerateContentConfig(
                        system_instruction=PROMPT_SANG_MATH,
                        temperature=0.1,
                        max_output_tokens=8192,
                        http_options=types.HttpOptions(timeout=50000)
                    )
                )
                txt = res.text or ''
                return txt.replace('```typst', '').replace('```', '').strip()
            except Exception as e:
                err_msg = str(e)
                if '429' in err_msg or 'RESOURCE_EXHAUSTED' in err_msg:
                    print(f' [{m} chạm quota 429, nghỉ 15s hồi lượt...]', end='', flush=True)
                    time.sleep(15)
                elif '503' in err_msg or 'UNAVAILABLE' in err_msg:
                    print(f' [{m} 503 bận, thử model tiếp theo...]', end='', flush=True)
                    time.sleep(3)
                    break
                else:
                    print(f' [{m} lỗi, thử lại...]', end='', flush=True)
                    time.sleep(3)
    return '// [Lỗi: Không thể nhận diện trang này]'

for pdf_path in all_pdfs:
    doc = fitz.open(pdf_path)
    fname = os.path.basename(pdf_path).replace('.pdf', '')
    print(f'\n🚀 Bắt đầu nhận diện: {fname}.pdf ({len(doc)} trang)...', flush=True)
    
    header = f'''// ================================================================
// ĐỀ THI TOÁN THPT - CHUẨN SANG-MATH:1.0.5
// Nhận diện tự động qua ConicTypst Kaggle Batch Engine
// ================================================================

#import "@preview/sang-math:1.0.5": *

#show: exam-theme.with(
  paper: "a4",
  cols: 2,
  theme: "ocean",
  show-header: true,
  show-footer: true,
  watermark: none,
  watermark-opacity: 0.05,
  school: [TRƯỜNG THPT],
  title: [{fname}],
  subject: [Môn: Toán - Thời gian: 90 phút],
  exam-id: "001"
)

'''
    typ_file = os.path.join('output_typst', f'{fname}.typ')
    with open(typ_file, 'w', encoding='utf-8') as f:
        f.write(header)
        
    for i, page in enumerate(doc):
        print(f'  -> Nhận diện trang {i+1}/{len(doc)}...', end='', flush=True)
        pix = page.get_pixmap(dpi=180)
        code = ocr_page_gemini(pix.tobytes('jpeg'))
        code = code.replace('#import "@preview/sang-math:1.0.5": *', '')
        page_content = f'\n// --- NỘI DUNG TỪ TRANG {i+1} ---\n' + code + '\n'
        
        with open(typ_file, 'a', encoding='utf-8') as f:
            f.write(page_content)
        print(' [XONG]', flush=True)
        
        # Nghỉ 4 giây giữa các trang để KHÔNG BỊ 429 trên các model 3.x Flash
        time.sleep(4)
        
    print(f'  ✅ Đã xuất Typst an toàn: {typ_file}', flush=True)
    
    docx_file = os.path.join('output_docx', f'{fname}.docx')
    try:
        subprocess.run([
            'pandoc', '-f', 'markdown', '-t', 'docx',
            typ_file, '-o', docx_file
        ], check=True, timeout=60)
        print(f'  ✅ Đã xuất Word DOCX: {docx_file}', flush=True)
    except Exception as err:
        pass

print('\n🎉 BƯỚC 3 ĐÃ HOÀN TẤT TẤT CẢ FILE ĐỀ THI!', flush=True)

In [ ]:
# ========================================================
# BƯỚC 4: Đóng gói và Tải toàn bộ kết quả về máy tính (1 Click)
# ========================================================
import zipfile, glob, os
from IPython.display import FileLink, display

zip_filename = 'dethi_conictypst_hoan_thanh.zip'
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for typ in glob.glob('output_typst/*.typ'):
        zipf.write(typ, os.path.join('Typst_Sources', os.path.basename(typ)))
    for docx in glob.glob('output_docx/*.docx'):
        zipf.write(docx, os.path.join('Word_Docx', os.path.basename(docx)))

print('\n=======================================================', flush=True)
print('🎉 TẤT CẢ ĐỀ THI ĐÃ ĐƯỢC XỬ LÝ XONG VÀ ĐÓNG GÓI THÀNH CÔNG!', flush=True)
print('👉 BẤM VÀO ĐƯỜNG LINK XANH DƯỚI ĐÂY ĐỂ TẢI FILE ZIP VỀ MÁY:', flush=True)
display(FileLink(zip_filename))